In [ ]:
import os
import shutil
import urllib.request, urllib.parse, urllib.error
from functools import reduce

import numpy as np
import pandas as pd

import plotly.express as px
import seaborn as sns
import matplotlib
from matplotlib import pyplot as plt

from tqdm.notebook import tqdm

In [ ]:
FILTERED_SUMMARY = '/mnt/craig/pan_phylon/S_pyogenes/metadata/filtered_species_summary.csv'
FILTERED_METADATA = '/mnt/craig/pan_phylon/S_pyogenes/metadata/filtered_species_metadata.csv'

In [ ]:
filtered_species_summary = pd.read_csv(FILTERED_SUMMARY, index_col=0, dtype='object')
filtered_species_metadata = pd.read_csv(FILTERED_METADATA, index_col=0, dtype='object')

# Ensure genome_id is a str
filtered_species_summary['genome_id'] = filtered_species_summary['genome_id'].astype('str')
filtered_species_metadata['genome_id'] = filtered_species_metadata['genome_id'].astype('str')

display(
    filtered_species_summary.shape,
    filtered_species_summary.head()
)

In [ ]:
complete_strains = filtered_species_metadata.genome_status == "Complete"


display(
    filtered_species_metadata[complete_strains].shape,
    filtered_species_metadata[complete_strains].head()
)


In [ ]:
download_paths = '/mnt/craig/pan_phylon/S_pyogenes/raw/genomes/'

In [ ]:
# Set of genome ids to download 
#dl_strains = set(filtered_species_summary.genome_id.astype(str))
dl_strains = set(filtered_species_metadata[complete_strains].genome_id.astype(str))

# Any previously downloaded strains in directory that do not meet curent filtration criteria
# This only matters when you are re-running this notebook/workflow
bad_dl = list()

for folder in os.listdir(download_paths):
    if folder in dl_strains:
        dl_strains.remove(str(folder))        
    else:
        bad_dl.append(str(folder))

In [ ]:
len(dl_strains) # if len is 0, all strains are already downloaded

In [ ]:
len(bad_dl) # if len is 0, no bad downloads need to be removed

In [ ]:
# delete bad_dl strains
# Only matters on a rerun
for folder in tqdm(bad_dl):
    print('removing folder:', folder)
    shutil.rmtree(os.path.join(download_paths, folder))

# Download (first-pass) filtered genomes

In [ ]:
import subprocess
import shutil
import os
from tqdm.auto import tqdm

VALID_BV_BRC_FILES = [
    'faa', 'features.tab', 'ffn', 'frn', 'gff',
    'pathway.tab', 'spgene.tab', 'subsystem.tab', 'fna'
]

def download_bv_brc_genomes(genomes, output_dir, filetypes=['fna', 'gff'], redownload=False):
    bad_genomes = []

    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Process filetypes
    source_target_filetypes = []
    for ftype in tqdm(filetypes, desc='Processing filetypes...'):
        if ftype in VALID_BV_BRC_FILES:
            ftype_source = 'PATRIC.' + ftype if ftype != 'fna' else ftype
            source_target_filetypes.append((ftype_source, ftype))

        elif ftype.replace('PATRIC.', '') in VALID_BV_BRC_FILES:
            source_target_filetypes.append((ftype, ftype.replace('PATRIC.', '')))

        else:
            print('Invalid filetype:', ftype)

    # Download relevant files
    for i, genome in tqdm(
        enumerate(genomes),
        desc='Downloading selected files...',
        total=len(genomes)
    ):
        genome_dir = os.path.join(output_dir, genome)
        os.makedirs(genome_dir, exist_ok=True)

        genome_base_url = f'ftps://ftp.bvbrc.org/genomes/{genome}/{genome}'
        genome_base_local = os.path.join(genome_dir, genome)

        genome_ok = True
        for source_filetype, target_filetype in source_target_filetypes:
            source = f'{genome_base_url}.{source_filetype}'
            target = f'{genome_base_local}.{target_filetype}'

            if os.path.exists(target) and not redownload:
                print(f'{i+1} Already exists: {target}')
                continue

            print(f'{i+1} {source} -> {target}')
            result = subprocess.run([
                'wget',
                '--ftp-user=anonymous',
                '--ftp-password=guest',
                '--secure-protocol=auto',
                '--no-check-certificate',
                '--continue',
                '-q',           # quiet; remove if you want wget progress
                '-O', target,
                source,
            ])

            if result.returncode != 0:
                print(f'Bad genome ID or missing file: {genome}')
                shutil.rmtree(genome_dir, ignore_errors=True)
                bad_genomes.append(genome)
                genome_ok = False
                break  # don't try remaining filetypes for this genome

        # Clean up empty dir if something failed mid-genome
        if not genome_ok:
            continue

    return bad_genomes

In [ ]:
bad_genomes = download_bv_brc_genomes(genomes=dl_strains, filetypes=['fna'], output_dir=download_paths, redownload=False)

In [ ]:
len(bad_genomes)

In [ ]:
dl_success = sorted(dl_strains - set(bad_genomes))
len(dl_success)

## Remove genomes with empty fasta files

In [ ]:
# some fasta files on BV-BRC are empty, remove those and remove them from metadata

## command to remove files with no text in them, list made by checking file lenth in terminal
## commented out since it has been run already

count = 0
for folder in os.listdir(download_paths):
    size = os.path.getsize(download_paths + '/' + folder + '/' + folder + '.fna' )
    if size == 0:
        count += 1
        bad_genomes.append(folder)
        print('Removing folder:', folder)
        shutil.rmtree(os.path.join(download_paths, folder))
print("Empty downloads total:", count)


# Update `summary` and `metadata` files

In [ ]:
filtered_species_metadata_complete = filtered_species_metadata[complete_strains].copy()
filtered_species_metadata_wgs = filtered_species_metadata[~complete_strains].copy()

display(
    filtered_species_metadata_complete.shape,
    filtered_species_metadata_wgs.shape
)

In [ ]:
downloaded_genomes = sorted(set(filtered_species_metadata_complete['genome_id']) - set(bad_genomes))

filtered_species_metadata_complete = (filtered_species_metadata_complete.
                                      drop_duplicates(subset=['genome_id']).
                                      set_index('genome_id').
                                      loc[downloaded_genomes].
                                      reset_index())

display(
    filtered_species_metadata_complete.shape,
    filtered_species_metadata_complete.head()
)

In [ ]:
# Common genomes list
common_genomes_complete = sorted(set(filtered_species_metadata_complete['genome_id']).intersection(set(filtered_species_summary['genome_id'])))
common_genomes_wgs = sorted(set(filtered_species_metadata_wgs['genome_id']).intersection(set(filtered_species_summary['genome_id'])))

# non-downloaded wgs genomes summary & metadata
filtered_species_metadata_wgs = filtered_species_metadata.set_index('genome_id').loc[common_genomes_wgs].reset_index()
filtered_species_summary_wgs = filtered_species_summary.set_index('genome_id').loc[common_genomes_wgs].reset_index()

# downloaded complete genomes summary & metadata
filtered_species_metadata = filtered_species_metadata_complete.set_index('genome_id').loc[common_genomes_complete].reset_index()
filtered_species_summary = filtered_species_summary.set_index('genome_id').loc[common_genomes_complete].reset_index()

display(
    filtered_species_summary.shape,
    filtered_species_summary.head(),
    filtered_species_metadata.shape,
    filtered_species_metadata.head(),
    'non-dowloaded summary & metadata',
    filtered_species_summary_wgs.shape,
    filtered_species_metadata_wgs.shape,
)

# Ensure complete sequences have parity with contig count

Complete sequences should not have more than 12 contigs total (1 main chromosome + 11 plasmids)

In [ ]:
(filtered_species_metadata.contigs.astype(int) > 11).sum() # number of strains with too many contigs

In [ ]:
# These strains need to be investigated
problem_strains = filtered_species_metadata.contigs.astype(int) > 11
filtered_species_metadata[problem_strains]

In [ ]:
# BioProject numbers don't line up with number of contigs, need to be removed
filtered_species_metadata[problem_strains][['genome_id', 'bioproject_accession', 'contigs']]

In [ ]:
filtered_species_metadata = filtered_species_metadata.set_index('genome_id').drop(labels=filtered_species_metadata[problem_strains].genome_id).reset_index()
filtered_species_summary = filtered_species_summary.set_index('genome_id').loc[filtered_species_metadata.genome_id.values].reset_index()

display(
    filtered_species_summary.shape,
    filtered_species_summary.head(),
    filtered_species_metadata.shape,
    filtered_species_metadata.head(),
)

In [ ]:
(filtered_species_metadata['genome_id'] == '1314.739').sum() # ensure reference strain is still part of the filtered & downloaded dataset

In [ ]:
# Filtered & downloaded summary
filepath = os.path.join(FILTERED_SUMMARY.split('filtered_species_summary.csv')[0], 'filtered_downloaded_species_summary.csv')
filtered_species_summary.to_csv(filepath)

In [ ]:
# Filtered & downloaded metadata
filepath = os.path.join(FILTERED_METADATA.split('filtered_species_metadata.csv')[0], 'filtered_downloaded_species_metadata.csv')
filtered_species_metadata.to_csv(filepath)

In [ ]:
# Filter & NOT downloaded summary & metadata
filepath = os.path.join(FILTERED_SUMMARY.split('filtered_species_summary.csv')[0], 'filtered_NOT_downloaded_species_summary.csv')
filtered_species_summary_wgs.to_csv(filepath)

filepath = os.path.join(FILTERED_METADATA.split('filtered_species_metadata.csv')[0], 'filtered_NOT_downloaded_species_metadata.csv')
filtered_species_metadata_wgs.to_csv(filepath)